<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/c4_ex12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Batch gradient descent wit early stopping for softmax regression without using Scikit-Learn, only NumPy. Use it on the iris dataset.
Unlike logistic regression where we have 2 classes (virginica vs not virginica) here we can have multiple (virginica, setosa, versicolor)

In [8]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
iris  = load_iris(as_frame=True)

iris.data.head(5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [11]:
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris["target"].values

We need to add the bias term for every instance ($x_0=1$). The easiest option to do this would be to use Scikit-Learn's add_dummy_feature().

In [12]:
X_with_bias = np.c_[np.ones(len(X)), X]

 Split in test and validation set without using Scikit-Learn's train_test_split()

In [13]:
test_ratio = 0.2
validation_ratio = 0.2
total_size = len(X_with_bias)

test_size = int(total_size * test_ratio)
val_size = int(total_size * validation_ratio)
train_size = total_size - val_size - test_size

# shuffle the indices
np.random.seed(42)
shuffle_indices = np.random.permutation(total_size)

X_train = X_with_bias[shuffle_indices[:train_size]]
y_train = y[shuffle_indices[:train_size]]
X_validation = X_with_bias[shuffle_indices[train_size:-test_size]]
y_validation = y[shuffle_indices[train_size:-test_size]]
X_test = X_with_bias[-test_size:]
y_test = y[-test_size:]


To train Softmax Regression, we must convert our class indices (0, 1, or 2) into one-hot encoded probability vectors where the true class is 1.0 and all others are 0.0. By creating an identity matrix with np.diag(np.ones(n)), we can use the class index array as a row selector to instantly pull out the correct one-hot vectors for every instance.

In [14]:
def to_one_hot(y):
    # Creates a 3x3 identity matrix a (which is a numpy array with 3 rows).
    # If we evaluate this array on the target y = [2, 0, 1, 2],
    # it uses advanced indexing to pull rows 2, 0, 1, and 2 in order:
    # Row 2 -> [0., 0., 1.]
    # Row 0 -> [1., 0., 0.]
    # Row 1 -> [0., 1., 0.]
    # Row 2 -> [0., 0., 1.]
    # a[y]
    # we get array([0., 0., 1.]
    #              [1., 0., 0.]
    #              [0., 1., 0.]
    #              [0., 0., 1.])
    # for each instance, the location of the 1 is precisely the class it belongs to
    return np.diag(np.ones(y.max() + 1))[y]

In [15]:
y_train[:10]

array([1, 0, 2, 1, 1, 0, 1, 2, 1, 1])

In [16]:
to_one_hot(y_train[:10])
# 1 is mapped to [0., 1., 0.], 2 to [0., 0., 1.], etc.

array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 0.]])

Looks good, so let's create the target class probabilities matrix for the training set and the test set:

In [18]:
Y_train_one_hot = to_one_hot(y_train)
Y_validation_one_hot = to_one_hot(y_validation)
Y_test_one_hot = to_one_hot(y_test)

Now let's scale the inputs. We compute the mean and standard deviation of each feature on the training set (except for the bias feature), then we center and scale each feature in the training set, the validation set, and the test set:


In [19]:
mean = X_train[:, 1:].mean(axis=0)
std = X_train[:, 1:].std(axis=0)

X_train[:, 1:] = (X_train[:, 1:] - mean) / std
X_validation[:, 1:] = (X_validation[:, 1:] - mean) / std

# we scale the test set using the training mean and std
X_test[:, 1:] = (X_test[:, 1:] - mean) / std

X_train[:10]
#note that each row of this matrix is one instance (now rescaled)

array([[ 1.        ,  0.63935691,  0.10418645],
       [ 1.        , -1.04480275, -1.06791114],
       [ 1.        ,  1.87440733,  1.53675017],
       [ 1.        ,  0.5270796 ,  0.49488565],
       [ 1.        ,  0.69549556,  0.36465258],
       [ 1.        , -1.15708007, -0.93767807],
       [ 1.        ,  0.0218317 ,  0.23441952],
       [ 1.        ,  0.86391153,  1.53675017],
       [ 1.        ,  0.5270796 ,  0.49488565],
       [ 1.        ,  0.19024767,  0.10418645]])

To understand what is going on with the dimensions of the vectors and stuff like this

In [20]:
# Take a sample instance matrix (5 instances, 3 features including bias)
X_sample = np.array([
    [1.0,  0.63935691,  0.10418645],  # Instance 0 (x)
    [1.0, -1.04480275, -1.06791114],
    [1.0,  1.87440733,  1.53675017],
    [1.0,  0.5270796,   0.49488565],
    [1.0,  0.69549556,  0.36465258]
])

# Randomly initialize the parameter matrix Theta (3 features x 3 classes)
np.random.seed(42)
Theta = np.random.randn(3, 3)

# Compute the full logits matrix for all instances and classes at once
logits = X_sample @ Theta

# --- TRACKING THE SINGLE SCALAR EXAMPLE ---
# Extract instance 0 row vector: x
x = X_sample[0]

# Extract class 0 weight column vector: theta_0
theta_0 = Theta[:, 0]

# Compute the scalar score s_0(x) using a dot product
s_0_x = x.dot(theta_0)

print(f"Instance vector x: {x}")
print(f"Weight vector theta_0: {theta_0}")
print(f"Calculated scalar score s_0(x): {s_0_x:.8f}")
print(f"Matches matrix value at logits[0, 0]: {logits[0, 0]:.8f}")

Instance vector x: [1.         0.63935691 0.10418645]
Weight vector theta_0: [0.49671415 1.52302986 1.57921282]
Calculated scalar score s_0(x): 1.63500639
Matches matrix value at logits[0, 0]: 1.63500639
